In [1]:
import pandas as pd
from sqlalchemy import create_engine
from DATA.stock_invest_function import *
import statsmodels.api as sm

In [2]:
start_date = '2015-01-01'
end_date = '2025-03-31'
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host': '192.168.0.230',
    'host': get_db_host(),         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

In [12]:
trade_df = fetch_table_data(db_info, "korea_monthly_trade_data_forecast")

# pivot table 생성
pivot_df = trade_df.pivot(index='date', columns='root_hs_code', values='final_expDlr_yoy')
pivot_df.index = pd.to_datetime(pivot_df.index)
export_yoy_data = pivot_df.loc[start_date:end_date]

✅ 'korea_monthly_trade_data_forecast' 테이블에서 168031건의 데이터를 가져왔습니다.


In [32]:
# KSE_Price
price_df = fetch_table_data(db_info, "KSE_Price")

# 0) 준비: date를 datetime으로
price_df = price_df.copy()
price_df['date'] = pd.to_datetime(price_df['date'])
price_df = price_df.sort_values(['code', 'date'])

# 1) 월말로 기록된(=각 월의 마지막 거래일) 데이터 추출
#    (그 달의 가장 늦은 날짜 한 건만 선택)
period = price_df['date'].dt.to_period('M')
last_idx = price_df.groupby(['code', period])['date'].idxmax()
month_end_df = price_df.loc[last_idx, ['date', 'code', 'close']].copy()

# 2) date를 '진짜 달력 월말'로 재부여
month_end_df['date'] = month_end_df['date'].dt.to_period('M').dt.to_timestamp('M')

# 3) 월말 데이터 피벗 및 3/6/12개월 수익률 계산
monthly_close = (
    month_end_df
      .pivot(index='date', columns='code', values='close')
      .sort_index()
)

# (선택) 인덱스를 ‘모든 월말’로 보정해 결측을 명확히 드러냄
full_idx = pd.date_range(monthly_close.index.min(), monthly_close.index.max(), freq='M')
monthly_close = monthly_close.reindex(full_idx)

# 수익률: (현재/과거) - 1  —— 월별 시계열이므로 periods=3/6/12 사용
ret_3m  = monthly_close.pct_change(3)
ret_6m  = monthly_close.pct_change(6)
ret_12m = monthly_close.pct_change(12)

✅ 'KSE_Price' 테이블에서 6280830건의 데이터를 가져왔습니다.


In [37]:
ret_3m_rssize = ret_3m.loc[start_date:end_date].dropna(axis=1, how='any')
ret_6m_rssize = ret_6m.loc[start_date:end_date].dropna(axis=1, how='any')
ret_12m_rssize = ret_12m.loc[start_date:end_date].dropna(axis=1, how='any')

In [56]:
def calc_correlation_matrix(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    """
    두 개의 DataFrame을 입력받아 (df1.columns × df2.columns) 상관계수 매트릭스를 반환.
    - 공통 인덱스 기준으로 정렬 후 상관계수 계산
    - 피어슨 상관계수 사용
    """
    # 1. 공통 인덱스 찾기
    common_index = df1.index.intersection(df2.index)

    # 2. 공통 인덱스만 추출
    df1_common = df1.loc[common_index]
    df2_common = df2.loc[common_index]

    # 3. 상관계수 매트릭스 계산
    corr_df = pd.DataFrame(
        index=df1_common.columns,
        columns=df2_common.columns,
        dtype=float
    )

    for col1 in df1_common.columns:
        for col2 in df2_common.columns:
            corr_df.loc[col1, col2] = df1_common[col1].corr(df2_common[col2])

    return corr_df


def melt_pivot(df, id_name="root_hs_code", var_name="hs_code", value_name="value",
               dropna=True, sort=True):
    # 인덱스명이 없다면 안전하게 이름을 부여
    idx_name = df.index.name or id_name
    long_df = (
        df.reset_index()  # 인덱스를 컬럼으로
          .melt(id_vars=[idx_name], var_name=var_name, value_name=value_name)  # 축소
    )
    if dropna:
        long_df = long_df.dropna(subset=[value_name])
    if sort:
        long_df = long_df.sort_values([idx_name, var_name]).reset_index(drop=True)
    return long_df



In [42]:
correlation_3m_result = calc_correlation_matrix(ret_3m_rssize, export_yoy_data)
correlation_12m_result = calc_correlation_matrix(ret_12m_rssize, export_yoy_data)

In [47]:
correlation_6m_result = calc_correlation_matrix(ret_6m_rssize, export_yoy_data)

In [60]:
long_3m_df = melt_pivot(correlation_3m_result, value_name="correlation")
long_6m_df = melt_pivot(correlation_6m_result, value_name="correlation")
long_12m_df = melt_pivot(correlation_12m_result, value_name="correlation")

long_3m_df['period'] = '3m'
long_6m_df['period'] = '6m'
long_12m_df['period'] = '12m'

correlation_result = pd.concat([long_3m_df, long_6m_df, long_12m_df])

In [3]:
# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# DB에 저장
correlation_result.to_sql(
    name="correlation_price_hscode",  # 테이블명
    con=engine,
    if_exists='replace',  # 'append'로 하면 기존 데이터 뒤에 추가
    index=False           # DataFrame 인덱스는 저장 안 함
)

print("✅ 데이터가 'correlation_price_hscode' 테이블에 저장되었습니다.")

NameError: name 'correlation_result' is not defined

In [3]:
correlation_table = fetch_table_data(db_info, "correlation_price_hscode")

correlation_table

✅ 'correlation_price_hscode' 테이블에서 3579023건의 데이터를 가져왔습니다.


,code,hs_code,correlation,period
0,000020,121120,-0.142576,3m
1,000020,1212,-0.183932,3m
2,000020,121221,-0.184513,3m
3,000020,151550,0.293330,3m
4,000020,151590,0.183652,3m
...,...,...,...,...
3579018,950130,940199,0.144811,12m
3579019,950130,940330,-0.211271,12m
3579020,950130,940540,0.145327,12m
3579021,950130,950300,0.072326,12m


In [4]:
# 1) period == '6m'만 필터
df6 = correlation_table[correlation_table['period'].eq('6m')].copy()

# (선택) 메모리/정렬 정리
df6['code'] = df6['code'].astype(str)
df6['hs_code'] = df6['hs_code'].astype(str)

# 2) 피벗: index=code, columns=hs_code, values=correlation
#    동일 쌍이 여러 개면 평균으로 집계(원하면 'first', 'max', 'median' 등으로 변경)
pivot_6m = df6.pivot_table(
    index='code',
    columns='hs_code',
    values='correlation',
    aggfunc='mean'   # 필요 시 변경 가능
)

# 보기 좋게 정렬(옵션)
pivot_6m = pivot_6m.sort_index().sort_index(axis=1)
pivot_6m.columns.name = None

# 결과 확인
print(pivot_6m.shape)
pivot_6m.head()

(1565, 763)


,121120,1212,121221,151550,151590,1518,170199,1902,190230,1905,...,903289,9301,9306,9401,940130,940199,940330,940540,950300,970191
code,,,,,,,,,,,,,,,,,,,,,
000020,-0.166224,-0.141201,-0.141067,0.200518,0.169332,0.053126,0.124689,0.135906,0.106678,0.227585,...,-0.052988,-0.047091,0.036441,-0.265584,0.166391,0.036558,-0.250435,0.180548,-0.234376,0.072149
000040,-0.092687,0.025891,0.029683,-0.020717,0.160230,0.002389,-0.058442,-0.091288,-0.112988,0.112714,...,0.254067,0.014769,-0.029816,0.206836,-0.009854,0.172702,-0.121542,0.235938,0.166468,-0.058583
000050,-0.079008,-0.167881,-0.169427,0.125020,0.402841,-0.026453,-0.032182,-0.080789,-0.102962,0.184362,...,-0.166606,-0.056094,0.096926,-0.100755,-0.088065,0.092070,-0.281213,0.092930,-0.023171,0.134164
000070,0.095991,-0.034173,-0.036399,-0.233401,0.373137,-0.048891,0.001752,-0.208513,-0.213591,0.078908,...,-0.063732,-0.094919,0.059610,0.078739,0.151587,-0.212850,-0.232816,0.156858,0.182762,0.304825
000080,-0.255229,-0.068073,-0.070641,0.220291,-0.053320,0.039329,-0.028526,0.015329,0.013433,0.006536,...,-0.179208,0.113880,0.182244,-0.266967,0.054604,0.310466,-0.162046,0.195534,-0.008970,-0.376207


In [5]:
pivot_6m.loc['012800'][['7404']]

7404    0.326845
Name: 012800, dtype: float64

In [6]:
# 1) period == '6m'만 필터
df12 = correlation_table[correlation_table['period'].eq('12m')].copy()

# (선택) 메모리/정렬 정리
df12['code'] = df12['code'].astype(str)
df12['hs_code'] = df12['hs_code'].astype(str)

# 2) 피벗: index=code, columns=hs_code, values=correlation
#    동일 쌍이 여러 개면 평균으로 집계(원하면 'first', 'max', 'median' 등으로 변경)
pivot_12m = df12.pivot_table(
    index='code',
    columns='hs_code',
    values='correlation',
    aggfunc='mean'   # 필요 시 변경 가능
)

# 보기 좋게 정렬(옵션)
pivot_12m = pivot_12m.sort_index().sort_index(axis=1)
pivot_12m.columns.name = None

# 결과 확인
print(pivot_6m.shape)
pivot_12m.head()

(1565, 763)


,121120,1212,121221,151550,151590,1518,170199,1902,190230,1905,...,903289,9301,9306,9401,940130,940199,940330,940540,950300,970191
code,,,,,,,,,,,,,,,,,,,,,
000020,-0.203235,-0.208487,-0.208391,0.120515,0.165130,-0.045137,0.061880,0.093765,0.050198,0.276168,...,0.111955,-0.079274,0.032839,-0.072862,0.087787,0.218426,-0.294988,0.313463,-0.105401,-0.009255
000040,-0.099725,-0.029640,-0.025506,-0.104833,0.118369,0.062155,-0.145168,-0.268164,-0.268194,0.034426,...,0.084373,0.000567,-0.083668,0.343249,0.045450,0.220647,-0.111417,0.286689,0.194623,-0.134765
000050,-0.082382,-0.163263,-0.162733,0.082269,0.654729,-0.104943,-0.080826,-0.264187,-0.285198,0.078676,...,-0.236647,-0.083475,0.054808,-0.120868,-0.099965,0.301415,-0.147949,0.115928,0.005883,-0.137076
000070,-0.068795,-0.045139,-0.045735,-0.153164,0.424908,-0.077803,-0.023381,-0.256953,-0.261974,0.085065,...,0.005314,-0.094957,0.030281,0.135865,0.117684,0.216339,-0.188323,0.267752,0.293836,-0.301729
000080,-0.275732,-0.190683,-0.193367,0.367065,-0.017842,0.107175,0.044439,0.165786,0.146204,0.177982,...,-0.167363,0.088698,0.221716,-0.411296,-0.027747,0.230057,-0.218748,0.293812,-0.154879,-0.218185


In [24]:
pivot_6m.loc['017510'][['853529']]

853529   -0.012352
Name: 017510, dtype: float64

In [7]:
pivot_12m[['854232']]

,854232
code,
000020,0.058473
000040,0.118246
000050,-0.034099
000070,0.028259
000080,-0.113240
...,...
900100,-0.100190
900110,-0.263997
900120,0.036439


In [ ]:
pivot_12m.loc['009540']